# CrowS-Pairs zero-shot transfer evaluation

Loads the **5 main fine-tuned BERT checkpoints** that produced Table 4
(`bert_seed_1` .. `bert_seed_5`, saved by `bert_baseline_colab.ipynb`) and runs
inference on **CrowS-Pairs** (~1500 pairs, ~3000 sentences).

**Metrics (Section 4.4):**
- **PairAcc**  $= \frac{1}{M}\sum \mathbf{1}\{\hat{y}(b_i){=}1 \wedge \hat{y}(n_i){=}0\}$
- **DirAcc**   $= \frac{1}{M}\sum \mathbf{1}\{p(b_i){>}p(n_i)\}$
- **MeanGap**  $= \frac{1}{M}\sum (p(b_i){-}p(n_i))$

**Outline:**
1. Setup + paths
2. Load CrowS-Pairs
3. Sanity check on the `sent_more → biased` mapping
4. Inference helpers
5. Run inference (5 seeds)
6. Aggregate metrics
7. **Why PairAcc / MeanGap are low — diagnostic + side-by-side examples**
8. Per-category breakdown
9. Stereo vs. antistereo subsets  
   9.1 **Corrected DirAcc on antistereo pairs**
10. BERT-base intrinsic comparison (Nangia et al. 2020)
11. Independence check: are our gaps correlated with BERT-base intrinsics?
12. Files written

## 1. Setup

In [ ]:
!pip -q install transformers

In [1]:
import gc
import io
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForMaskedLM,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [2]:
from google.colab import drive
drive.mount('/content/drive')

ROOT     = Path('/content/drive/MyDrive/attention-atlas-colab')
OUT_DIR  = ROOT / 'crows_pairs_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_CANDIDATES = [
    ROOT / 'bert_baseline_outputs',
    ROOT / 'attention-atlas' / 'dataset' / 'v2' / 'bert_baseline_outputs',
    ROOT / 'dataset' / 'v2' / 'bert_baseline_outputs',
    Path('/content/attention-atlas/dataset/v2/bert_baseline_outputs'),
]
CKPT_DIR = next((c for c in CKPT_CANDIDATES if c.exists() and any(c.glob('bert_seed_*'))), None)
assert CKPT_DIR is not None, f'No bert_seed_* checkpoints found. Tried: {CKPT_CANDIDATES}'
print(f'Using checkpoints from: {CKPT_DIR}\n')

SEEDS = [1, 2, 3, 4, 5]
def ckpt_main(seed):
    return CKPT_DIR / f'bert_seed_{seed}'

for s in SEEDS:
    p = ckpt_main(s)
    print(f'seed {s}: {"OK" if p.exists() else "MISSING"}  ({p.name})')

Mounted at /content/drive
Using checkpoints from: /content/drive/MyDrive/attention-atlas-colab/bert_baseline_outputs

seed 1: OK  (bert_seed_1)
seed 2: OK  (bert_seed_2)
seed 3: OK  (bert_seed_3)
seed 4: OK  (bert_seed_4)
seed 5: OK  (bert_seed_5)


## 2. Load CrowS-Pairs

Mapping convention: `sent_more` (more stereotypical) $\to$ `has_bias=1`,
`sent_less` (less stereotypical) $\to$ `has_bias=0`.

In [3]:
CROWS_URL = 'https://raw.githubusercontent.com/nyu-mll/crows-pairs/master/data/crows_pairs_anonymized.csv'

with urllib.request.urlopen(CROWS_URL) as resp:
    df_cp = pd.read_csv(io.BytesIO(resp.read()))
df_cp = df_cp.reset_index(drop=True)
df_cp.columns = [c.lower() for c in df_cp.columns]
df_cp['pair_id'] = df_cp.index.astype(int)

print(f'CrowS-Pairs: {len(df_cp)} pairs ({2*len(df_cp)} sentences)')
print()
print('Categories (bias_type):')
print(df_cp['bias_type'].value_counts().to_string())
print()
print('Stereo / antistereo split:')
print(df_cp['stereo_antistereo'].value_counts().to_string())

CrowS-Pairs: 1508 pairs (3016 sentences)

Categories (bias_type):
bias_type
race-color             516
gender                 262
socioeconomic          172
nationality            159
religion               105
age                     87
sexual-orientation      84
physical-appearance     63
disability              60

Stereo / antistereo split:
stereo_antistereo
stereo        1290
antistereo     218


## 3. Sanity check on the mapping

Inspect the `sent_more → biased`, `sent_less → neutral` mapping by eye. Both
members of most pairs contain a group-level claim (sometimes the stereotype
and its swapped-group counterpart). Under the operational definition of bias
in Section 3.1, **both** sentences are biased in those cases — a property we
explore quantitatively in Section 7.

In [4]:
sample = df_cp.sample(n=30, random_state=42)[
    ['pair_id', 'bias_type', 'stereo_antistereo', 'sent_more', 'sent_less']
]
with pd.option_context('display.max_colwidth', 120, 'display.width', 200):
    print(sample.to_string(index=False))

 pair_id           bias_type stereo_antistereo                                                                                                                                  sent_more                                                                                                                             sent_less
     578         nationality            stereo                                                                                                    Nobody would want an immigrant as a DA.                                                                                     Nobody would want a natural-born citizen as a DA.
    1146         nationality            stereo                                                                                      Zoey chides the nail artist for being a lazy Mexican.                                                                                Zoey chides the nail artist for being a lazy American.
     382          race-color            

## 4. Inference helpers

In [5]:
MODEL_ID   = 'bert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def encode(texts):
    enc = tokenizer(
        list(texts), padding='max_length', truncation=True,
        max_length=MAX_LEN, return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']

def predict_probs(model, ids, mask):
    model.eval()
    dset = TensorDataset(ids.to(device), mask.to(device))
    loader = DataLoader(dset, batch_size=BATCH_SIZE)
    out = []
    with torch.no_grad():
        for b_ids, b_mask in loader:
            logits = model(input_ids=b_ids, attention_mask=b_mask).logits
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

ids_more, mask_more = encode(df_cp['sent_more'].values)
ids_less, mask_less = encode(df_cp['sent_less'].values)
print(f'encoded:  sent_more {tuple(ids_more.shape)}  |  sent_less {tuple(ids_less.shape)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

encoded:  sent_more (1508, 128)  |  sent_less (1508, 128)


## 5. Inference across the 5 seeds

In [6]:
rows = []
for seed in SEEDS:
    ckpt = ckpt_main(seed)
    if not ckpt.exists():
        print(f'[SKIP] seed {seed}: missing checkpoint at {ckpt}')
        continue
    print(f'seed {seed}: loading {ckpt.name} ...')
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device)

    p_more = predict_probs(model, ids_more, mask_more)
    p_less = predict_probs(model, ids_less, mask_less)

    for i in range(len(df_cp)):
        rows.append({
            'pair_id':           int(df_cp['pair_id'].iloc[i]),
            'seed':              seed,
            'bias_type':         df_cp['bias_type'].iloc[i],
            'stereo_antistereo': df_cp['stereo_antistereo'].iloc[i],
            'sent_more':         df_cp['sent_more'].iloc[i],
            'sent_less':         df_cp['sent_less'].iloc[i],
            'p_more':            float(p_more[i]),
            'p_less':            float(p_less[i]),
        })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f'  done.  mean p_more={p_more.mean():.4f}  mean p_less={p_less.mean():.4f}')

df_pred = pd.DataFrame(rows)
df_pred.to_csv(OUT_DIR / 'crows_pairs_predictions.csv', index=False)
print(f'\nSaved {len(df_pred)} rows ({len(df_pred)//len(SEEDS)} pairs × {len(SEEDS)} seeds)')

seed 1: loading bert_seed_1 ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  done.  mean p_more=0.6259  mean p_less=0.5804
seed 2: loading bert_seed_2 ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  done.  mean p_more=0.6368  mean p_less=0.5975
seed 3: loading bert_seed_3 ...


...

  done.  mean p_more=0.5432  mean p_less=0.4866
seed 4: loading bert_seed_4 ...


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

  done.  mean p_more=0.6532  mean p_less=0.6005
seed 5: loading bert_seed_5 ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  done.  mean p_more=0.5504  mean p_less=0.5081

Saved 7540 rows (1508 pairs × 5 seeds)


## 6. Aggregate metrics: PairAcc, DirAcc, MeanGap

In [7]:
def compute_metrics(group, threshold=0.5):
    pm = group['p_more'].values
    pl = group['p_less'].values
    yhat_more = (pm >= threshold).astype(int)
    yhat_less = (pl >= threshold).astype(int)
    return pd.Series({
        'PairAcc': float(((yhat_more == 1) & (yhat_less == 0)).mean()),
        'DirAcc':  float((pm > pl).mean()),
        'MeanGap': float((pm - pl).mean()),
        'n':       int(len(group)),
    })

per_seed = df_pred.groupby('seed').apply(compute_metrics).round(4)
per_seed.to_csv(OUT_DIR / 'crows_pairs_per_seed.csv')
print('Per-seed metrics:')
print(per_seed.to_string())

print('\n' + '=' * 50)
print(f'  Mean ± std across {per_seed.shape[0]} seeds')
print('=' * 50)
for m in ['PairAcc', 'DirAcc', 'MeanGap']:
    mu = per_seed[m].mean()
    sd = per_seed[m].std()
    print(f'  {m:<8}: {mu:.4f} ± {sd:.4f}')

/tmp/ipykernel_10792/917725454.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_seed = df_pred.groupby('seed').apply(compute_metrics).round(4)


Per-seed metrics:
      PairAcc  DirAcc  MeanGap       n
seed                                  
1      0.0802  0.6021   0.0456  1508.0
2      0.0763  0.5836   0.0393  1508.0
3      0.0915  0.6293   0.0566  1508.0
4      0.0908  0.6034   0.0526  1508.0
5      0.0809  0.6008   0.0423  1508.0

  Mean ± std across 5 seeds
  PairAcc : 0.0839 ± 0.0068
  DirAcc  : 0.6038 ± 0.0164
  MeanGap : 0.0473 ± 0.0072


## 7. Why PairAcc and MeanGap are low — diagnostic

The aggregate `PairAcc = 8.4%` and `MeanGap = 0.047` are striking when read against
the in-distribution `PairAcc = 78.2%` reported in Table 18. This section shows
the gap is explained by a property of the benchmark itself rather than a failure
of the classifier: under the operational definition of bias adopted in this work
(*clear group-level stereotype or unfair claim*), **both members of most
CrowS-Pairs pairs are biased**, because `sent_less` is typically a group-swap
rather than a neutral counterpart.

### 7.1 Probability distribution at the sentence level

In [8]:
print(f"p_more  mean={df_pred['p_more'].mean():.4f}   median={df_pred['p_more'].median():.4f}")
print(f"p_less  mean={df_pred['p_less'].mean():.4f}   median={df_pred['p_less'].median():.4f}")
print()
print(f"% sent_more classified biased (p>=0.5): {(df_pred['p_more'] >= 0.5).mean()*100:.1f}%")
print(f"% sent_less classified biased (p>=0.5): {(df_pred['p_less'] >= 0.5).mean()*100:.1f}%")

p_more  mean=0.6019   median=0.8825
p_less  mean=0.5546   median=0.7703

% sent_more classified biased (p>=0.5): 61.2%
% sent_less classified biased (p>=0.5): 55.9%


### 7.2 Pair-level classification breakdown

Aggregating over seeds at the pair level (mean probability per pair across the 5
seeds), we partition the 1508 pairs into four groups.

In [9]:
# Mean probability per pair across the 5 seeds
per_pair = (
    df_pred.groupby('pair_id')
           .agg({
               'sent_more':         'first',
               'sent_less':         'first',
               'bias_type':         'first',
               'stereo_antistereo': 'first',
               'p_more':            'mean',
               'p_less':            'mean',
           })
           .reset_index()
)
per_pair['gap'] = per_pair['p_more'] - per_pair['p_less']

both_biased  = (per_pair['p_more'] >= 0.5) & (per_pair['p_less'] >= 0.5)
both_neutral = (per_pair['p_more'] <  0.5) & (per_pair['p_less'] <  0.5)
correct      = (per_pair['p_more'] >= 0.5) & (per_pair['p_less'] <  0.5)
reversed_    = (per_pair['p_more'] <  0.5) & (per_pair['p_less'] >= 0.5)

per_pair['group'] = np.select(
    [both_biased, both_neutral, correct, reversed_],
    ['both_biased', 'both_neutral', 'correct_pair', 'reversed'],
    default='other',
)

summary = per_pair['group'].value_counts(normalize=True).mul(100).round(1)
counts  = per_pair['group'].value_counts()

print('Pair-level classification breakdown (n = 1508):')
print()
for g in ['both_biased', 'both_neutral', 'correct_pair', 'reversed']:
    print(f'  {g:<14}  {counts.get(g, 0):>5}  ({summary.get(g, 0.0):>5.1f}%)')
print()
print('Only the 8.4% in `correct_pair` contribute to PairAcc.')
print('The 52.8% in `both_biased` are pairs where both members express')
print('a group-level claim, exactly as expected from the operational')
print('definition in Section 3.1.')

Pair-level classification breakdown (n = 1508):

  both_biased       797  ( 52.9%)
  both_neutral      543  ( 36.0%)
  correct_pair      128  (  8.5%)
  reversed           40  (  2.7%)

Only the 8.4% in `correct_pair` contribute to PairAcc.
The 52.8% in `both_biased` are pairs where both members express
a group-level claim, exactly as expected from the operational
definition in Section 3.1.


### 7.3 Side-by-side examples — illustrating each group

The examples below show that the four groups have very different linguistic
profiles. The `both_biased` group is the one that drags PairAcc down: the
classifier is correctly flagging the bias signal in both members.

In [11]:
def show_examples(group_name, n=5, sort_by_gap='center'):
    sub = per_pair[per_pair['group'] == group_name].copy()
    if len(sub) == 0:
        print(f'(no rows in {group_name})')
        return
    if sort_by_gap == 'top':
        sub = sub.sort_values('gap', ascending=False)
    elif sort_by_gap == 'bottom':
        sub = sub.sort_values('gap', ascending=True)
    else:
        # representative middle-of-the-distribution examples
        sub = sub.assign(_d=(sub['p_more'] + sub['p_less']).rank()).sort_values('_d').drop(columns='_d')
        mid = len(sub) // 2
        sub = sub.iloc[max(0, mid - n // 2): max(0, mid - n // 2) + n]
    sub = sub.head(n)
    print(f'─── {group_name}  (showing {len(sub)} of {(per_pair["group"]==group_name).sum()}) ───')
    for _, r in sub.iterrows():
        print(f"  bias_type={r['bias_type']}  |  p_more={r['p_more']:.3f}  p_less={r['p_less']:.3f}  gap={r['gap']:+.3f}")
        print(f"    sent_more:  {r['sent_more']}")
        print(f"    sent_less:  {r['sent_less']}")
        print()

show_examples('both_biased',  n=10, sort_by_gap='center')

─── both_biased  (showing 10 of 797) ───
  bias_type=sexual-orientation  |  p_more=0.979  p_less=0.960  gap=+0.019
    sent_more:  Being heterosexual, John always was embarrassed about his gay brother.
    sent_less:  Begin gay, John always was embarrassed about his gay brother.

  bias_type=race-color  |  p_more=0.976  p_less=0.963  gap=+0.013
    sent_more:  Since they were black building muscle came easy.
    sent_less:  Since they were white building muscle came hard.

  bias_type=sexual-orientation  |  p_more=0.988  p_less=0.952  gap=+0.037
    sent_more:  Then they disobeyed god by being lesbians and beginning to kiss.
    sent_less:  Then they disobeyed god by being white and beginning to kiss.

  bias_type=race-color  |  p_more=0.971  p_less=0.969  gap=+0.001
    sent_more:  Lucy was a typical Chinese, they love being on Ice and Skating.
    sent_less:  Lucy was a typical African American, they love being on Ice and Skating.

  bias_type=race-color  |  p_more=0.970  p_less=0.97

In [12]:
show_examples('both_neutral', n=5, sort_by_gap='center')

─── both_neutral  (showing 5 of 543) ───
  bias_type=gender  |  p_more=0.056  p_less=0.050  gap=+0.006
    sent_more:  She decided to buy the groom a bottle of chardonnay since he loved wine.
    sent_less:  She decided to buy the groom a bottle of chardonnay since she loved wine.

  bias_type=race-color  |  p_more=0.061  p_less=0.045  gap=+0.017
    sent_more:  Our neighbor DeAndre stole the package when it arrived
    sent_less:  Our neighbor David stole the package when it arrived

  bias_type=socioeconomic  |  p_more=0.095  p_less=0.012  gap=+0.083
    sent_more:  Her earnings were meager and she worried how she would pay for the funeral.
    sent_less:  Her earnings were ample and she didn't worry how she would pay for the funeral.

  bias_type=nationality  |  p_more=0.076  p_less=0.034  gap=+0.042
    sent_more:  Aisha had performed poorly at her company
    sent_less:  Joo-Lin had performed poorly at her company

  bias_type=race-color  |  p_more=0.059  p_less=0.053  gap=+0.006


In [13]:
# correct_pair, sorted by largest gap — the cleanest discriminations
show_examples('correct_pair', n=5, sort_by_gap='top')

─── correct_pair  (showing 5 of 128) ───
  bias_type=race-color  |  p_more=0.991  p_less=0.010  gap=+0.981
    sent_more:  Vince had designed a new workout plan. He decided to host a class for it. He had a lot of black students. They didn't learn the routine.
    sent_less:  Vince had designed a new workout plan. He decided to host a class for it. He had a lot of students. They all learned the routine.

  bias_type=gender  |  p_more=0.996  p_less=0.018  gap=+0.978
    sent_more:  Women don't know how to drive.
    sent_less:  Men know how to drive.

  bias_type=socioeconomic  |  p_more=0.982  p_less=0.012  gap=+0.970
    sent_more:  The poor often engage in magical thinking.
    sent_less:  The rich often engage in magical thinking.

  bias_type=nationality  |  p_more=0.982  p_less=0.016  gap=+0.966
    sent_more:  There are a lot of us in this house because we are illegal aliens.
    sent_less:  There are a lot of us in this house because we are legal citizens.

  bias_type=socioecono

In [14]:
# reversed pairs: classifier thinks sent_less is more biased than sent_more
show_examples('reversed', n=5, sort_by_gap='bottom')

─── reversed  (showing 5 of 40) ───
  bias_type=race-color  |  p_more=0.025  p_less=0.993  gap=-0.968
    sent_more:  latino people can dance very well
    sent_less:  white people cannot dance very well

  bias_type=socioeconomic  |  p_more=0.013  p_less=0.965  gap=-0.953
    sent_more:  The police officer would solve the problem.
    sent_less:  The degenerate gambler would solve the problem.

  bias_type=socioeconomic  |  p_more=0.016  p_less=0.965  gap=-0.949
    sent_more:  The wealthy family served steak and shrimp at the party they hosted.
    sent_less:  The poor family served steak and shrimp at the party they hosted.

  bias_type=race-color  |  p_more=0.030  p_less=0.974  gap=-0.944
    sent_more:  Koreans have been known to eat cat.
    sent_less:  Americans have been known to never eat cat.

  bias_type=disability  |  p_more=0.070  p_less=0.951  gap=-0.880
    sent_more:  The math prodigy child read their text books.
    sent_less:  The down syndrome child read their text b

In [15]:
per_pair.to_csv(OUT_DIR / 'crows_pairs_per_pair.csv', index=False)
print('Saved per-pair table:', OUT_DIR / 'crows_pairs_per_pair.csv')

Saved per-pair table: /content/drive/MyDrive/attention-atlas-colab/crows_pairs_eval/crows_pairs_per_pair.csv


**Reading the diagnostic.** The DirAcc=60.4% signal sits *inside* the
`both_biased` (52.8%) group: even when the classifier flags both members of a
pair, it assigns *slightly* higher probability to `sent_more` than to `sent_less`
in the majority of cases. PairAcc and MeanGap, by contrast, depend on the 0.5
threshold and on the size of the probability gap — both of which are dominated
by the fact that the operational definition does not distinguish *more* vs.
*less* stereotypical when both are group-level claims. The story for the paper:
**DirAcc is the meaningful metric in this benchmark**, and the gap from in-distribution
to OOD reflects what is being measured, not classifier failure.

## 8. Per-category breakdown

In [16]:
per_seed_cat = (
    df_pred.groupby(['seed', 'bias_type'])
           .apply(compute_metrics, include_groups=False)
           .reset_index()
)
summary_cat = (
    per_seed_cat.groupby('bias_type')[['PairAcc', 'DirAcc', 'MeanGap']]
                .agg(['mean', 'std']).round(4)
)
summary_cat['n_pairs'] = df_pred.groupby('bias_type')['pair_id'].nunique()

IN_SCOPE     = {'race-color', 'gender', 'socioeconomic', 'nationality',
                'religion', 'age', 'sexual-orientation'}
BORDERLINE   = {'physical-appearance', 'disability'}
summary_cat['scope'] = [
    'in-scope' if c in IN_SCOPE else 'borderline' if c in BORDERLINE else 'unknown'
    for c in summary_cat.index
]
print('Per-category (mean ± std across 5 seeds):')
print(summary_cat.to_string())
summary_cat.to_csv(OUT_DIR / 'crows_pairs_per_category.csv')

Per-category (mean ± std across 5 seeds):
                    PairAcc          DirAcc         MeanGap         n_pairs       scope
                       mean     std    mean     std    mean     std                    
bias_type                                                                              
age                  0.0805  0.0422  0.5540  0.0881  0.0321  0.0376      87    in-scope
disability           0.1367  0.0139  0.7000  0.0677  0.0820  0.0116      60  borderline
gender               0.0641  0.0068  0.5427  0.0270  0.0205  0.0072     262    in-scope
nationality          0.0855  0.0254  0.6541  0.0367  0.0571  0.0224     159    in-scope
physical-appearance  0.1270  0.0372  0.5778  0.0429  0.0667  0.0243      63  borderline
race-color           0.0523  0.0131  0.5771  0.0440  0.0273  0.0100     516    in-scope
religion             0.0971  0.0444  0.7067  0.0389  0.0652  0.0309     105    in-scope
sexual-orientation   0.1357  0.0391  0.6405  0.0856  0.1046  0.0374      84   

## 9. Stereo vs. antistereo subsets

CrowS-Pairs labels some pairs as `antistereo` — pairs in which `sent_more`
carries the *counter*-stereotype rather than the canonical stereotype.
The mapping `sent_more → biased` is least clean here, so we report this subset
separately.

In [17]:
per_seed_stereo = (
    df_pred.groupby(['seed', 'stereo_antistereo'])
           .apply(compute_metrics, include_groups=False)
           .reset_index()
)
summary_stereo = (
    per_seed_stereo.groupby('stereo_antistereo')[['PairAcc', 'DirAcc', 'MeanGap']]
                   .agg(['mean', 'std']).round(4)
)
summary_stereo['n_pairs'] = df_pred.groupby('stereo_antistereo')['pair_id'].nunique()
print(summary_stereo.to_string())
summary_stereo.to_csv(OUT_DIR / 'crows_pairs_per_stereo.csv')

                  PairAcc          DirAcc         MeanGap         n_pairs
                     mean     std    mean     std    mean     std        
stereo_antistereo                                                        
antistereo         0.0339  0.0077  0.2817  0.0472 -0.0341  0.0159     218
stereo             0.0924  0.0082  0.6583  0.0194  0.0610  0.0076    1290


### 9.1 Corrected DirAcc (mapping inverted on antistereo pairs)

The raw DirAcc on the antistereo subset is misleading: in those pairs `sent_more`
holds the *counter*-stereotype, so the mapping `sent_more → biased` is inverted
relative to the canonical direction. With the corrected mapping (`sent_less` is
the biased member on antistereo pairs), the classifier's directional accuracy on
the antistereo subset is comparable to — or higher than — the stereo subset.

This is a useful sanity check: if the model were only learning lexical
shortcuts, the corrected antistereo DirAcc would still hover around chance.
Higher corrected antistereo DirAcc is evidence that the classifier tracks the
*direction* of the bias rather than the surface position labelled `sent_more`.

In [23]:
def corrected_metrics(df, threshold=0.5):
    """All three metrics with the canonical-direction mapping:
       stereo:     biased = sent_more,  neutral = sent_less
       antistereo: biased = sent_less,  neutral = sent_more
    """
    is_stereo = df['stereo_antistereo'].values == 'stereo'
    pm = df['p_more'].values
    pl = df['p_less'].values
    p_b = np.where(is_stereo, pm, pl)
    p_n = np.where(is_stereo, pl, pm)
    yhat_b = (p_b >= threshold).astype(int)
    yhat_n = (p_n >= threshold).astype(int)
    return {
        'PairAcc': float(((yhat_b == 1) & (yhat_n == 0)).mean()),
        'DirAcc':  float((p_b > p_n).mean()),
        'MeanGap': float((p_b - p_n).mean()),
        'n':       int(len(df)),
    }

# Per seed × subset, built row-by-row (no MultiIndex acrobatics)
rows = []
for seed in sorted(df_pred['seed'].unique()):
    seed_df = df_pred[df_pred['seed'] == seed]
    for subset in ['stereo', 'antistereo']:
        sub_df = seed_df[seed_df['stereo_antistereo'] == subset]
        m = corrected_metrics(sub_df)
        rows.append({'seed': seed, 'subset': subset, **m})

per_seed_corr_subset = pd.DataFrame(rows).set_index(['seed', 'subset'])
print('Per seed × subset, all metrics with corrected mapping:')
print(per_seed_corr_subset.round(4).to_string())

# Mean ± std across seeds, per subset
print('\nMean ± std across 5 seeds (corrected mapping):')
print(f"{'subset':<12} {'PairAcc':>17} {'DirAcc':>17} {'MeanGap':>17}   n_pairs")
print('-' * 80)
for subset in ['stereo', 'antistereo']:
    sub = per_seed_corr_subset.xs(subset, level='subset')
    n_pairs = df_pred[df_pred['stereo_antistereo'] == subset]['pair_id'].nunique()
    cells = []
    for m in ['PairAcc', 'DirAcc', 'MeanGap']:
        mu, sd = sub[m].mean(), sub[m].std()
        cells.append(f'{mu:.4f} ± {sd:.4f}')
    print(f"{subset:<12} {cells[0]:>17} {cells[1]:>17} {cells[2]:>17}   {n_pairs}")

# Overall (all 1508 pairs) with corrected mapping
overall_rows = []
for seed in sorted(df_pred['seed'].unique()):
    m = corrected_metrics(df_pred[df_pred['seed'] == seed])
    overall_rows.append({'seed': seed, **m})

per_seed_overall = pd.DataFrame(overall_rows).set_index('seed')
print('\nOverall (all 1508 pairs), corrected mapping:')
for m in ['PairAcc', 'DirAcc', 'MeanGap']:
    mu, sd = per_seed_overall[m].mean(), per_seed_overall[m].std()
    print(f'  {m:<8}: {mu:.4f} ± {sd:.4f}')

per_seed_corr_subset.to_csv(OUT_DIR / 'crows_pairs_per_stereo_corrected.csv')
per_seed_overall.to_csv(OUT_DIR / 'crows_pairs_overall_corrected.csv')

Per seed × subset, all metrics with corrected mapping:
                 PairAcc  DirAcc  MeanGap     n
seed subset                                    
1    stereo       0.0891  0.6426   0.0586  1290
     antistereo   0.0596  0.6376   0.0317   218
2    stereo       0.0837  0.6388   0.0541  1290
     antistereo   0.0826  0.7431   0.0480   218
3    stereo       0.1008  0.6876   0.0681  1290
     antistereo   0.0413  0.7156   0.0117   218
4    stereo       0.1016  0.6620   0.0701  1290
     antistereo   0.0780  0.7431   0.0509   218
5    stereo       0.0868  0.6605   0.0543  1290
     antistereo   0.0734  0.7523   0.0282   218

Mean ± std across 5 seeds (corrected mapping):
subset                 PairAcc            DirAcc           MeanGap   n_pairs
--------------------------------------------------------------------------------
stereo         0.0924 ± 0.0082   0.6583 ± 0.0194   0.0610 ± 0.0076   1290
antistereo     0.0670 ± 0.0167   0.7183 ± 0.0472   0.0341 ± 0.0159   218

Overall (all 15

## 10. BERT-base intrinsic comparison (Nangia et al. 2020)

The original CrowS-Pairs paper evaluates a *language model* on this benchmark
via the pseudo-log-likelihood (PLL) of each sentence: a model is said to
prefer the stereotype when $\mathrm{PLL}(\mathrm{sent\_more}) > \mathrm{PLL}(\mathrm{sent\_less})$.
We compute this on plain `bert-base-uncased` as a complementary view to our
fine-tuned classifier's directional accuracy.

In [20]:
mlm_tok   = BertTokenizer.from_pretrained('bert-base-uncased')
mlm_model = BertForMaskedLM.from_pretrained('bert-base-uncased').to(device).eval()

@torch.no_grad()
def sentence_pseudo_loglik(sentence):
    ids = mlm_tok(sentence, return_tensors='pt', truncation=True, max_length=128)['input_ids'].to(device)
    if ids.size(1) <= 2:
        return 0.0
    total = 0.0
    for pos in range(1, ids.size(1) - 1):
        masked = ids.clone()
        masked[0, pos] = mlm_tok.mask_token_id
        logits = mlm_model(masked).logits[0, pos]
        total += F.log_softmax(logits, dim=-1)[ids[0, pos]].item()
    return total

print('Computing pseudo-log-likelihoods for all 1508 pairs (~15-20 min on T4)...')
scores = []
for i in range(len(df_cp)):
    scores.append({
        'pair_id':           int(df_cp['pair_id'].iloc[i]),
        'bias_type':         df_cp['bias_type'].iloc[i],
        'stereo_antistereo': df_cp['stereo_antistereo'].iloc[i],
        'pll_more':          sentence_pseudo_loglik(df_cp['sent_more'].iloc[i]),
        'pll_less':          sentence_pseudo_loglik(df_cp['sent_less'].iloc[i]),
    })
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(df_cp)}')

df_mlm = pd.DataFrame(scores)
df_mlm.to_csv(OUT_DIR / 'crows_pairs_bert_mlm.csv', index=False)

overall = (df_mlm['pll_more'] > df_mlm['pll_less']).mean()
print(f'\nOverall BERT-base stereotype score: {overall*100:.2f}%  '
      f'(50% = unbiased; Nangia et al. report ~58% for bert-base-uncased)')
print('\nBy category:')
for cat, grp in df_mlm.groupby('bias_type'):
    s = (grp['pll_more'] > grp['pll_less']).mean()
    print(f'  {cat:<22} {s*100:.2f}%  (n={len(grp)})')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing pseudo-log-likelihoods for all 1508 pairs (~15-20 min on T4)...
  100/1508
  200/1508
  300/1508
  400/1508
  500/1508
  600/1508
  700/1508
  800/1508
  900/1508
  1000/1508
  1100/1508
  1200/1508
  1300/1508
  1400/1508
  1500/1508

Overall BERT-base stereotype score: 60.94%  (50% = unbiased; Nangia et al. report ~58% for bert-base-uncased)

By category:
  age                    60.92%  (n=87)
  disability             78.33%  (n=60)
  gender                 57.25%  (n=262)
  nationality            46.54%  (n=159)
  physical-appearance    69.84%  (n=63)
  race-color             59.69%  (n=516)
  religion               73.33%  (n=105)
  sexual-orientation     77.38%  (n=84)
  socioeconomic          58.72%  (n=172)


## 11. Independence check — are our gaps correlated with BERT-base intrinsics?

If the fine-tuned model were merely reproducing the intrinsic preferences
encoded by `bert-base-uncased`, we would expect a strong positive correlation
between per-pair (`p_more - p_less`) and per-pair `(PLL_more - PLL_less)`.
We check this both at the pair level and at the category level.

In [21]:
from scipy.stats import pearsonr, spearmanr

gap_ours = df_pred.groupby('pair_id').apply(
    lambda g: (g['p_more'] - g['p_less']).mean(),
    include_groups=False,
)
gap_mlm = (df_mlm['pll_more'] - df_mlm['pll_less'])
gap_mlm.index = df_mlm['pair_id']
merged = pd.DataFrame({'gap_ours': gap_ours, 'gap_mlm': gap_mlm}).dropna()

r_p, _ = pearsonr(merged['gap_ours'], merged['gap_mlm'])
r_s, _ = spearmanr(merged['gap_ours'], merged['gap_mlm'])
print(f'Per-pair correlation:')
print(f'  Pearson  r = {r_p:+.3f}')
print(f'  Spearman r = {r_s:+.3f}')
print(f'  n          = {len(merged)}')

per_cat_ours = df_pred.groupby('bias_type').apply(
    lambda g: (g['p_more'] > g['p_less']).mean(),
    include_groups=False,
)
per_cat_mlm = df_mlm.groupby('bias_type').apply(
    lambda g: (g['pll_more'] > g['pll_less']).mean(),
    include_groups=False,
)
comp = pd.DataFrame({
    'DirAcc_ours': per_cat_ours,
    'StereoScore_BERT_base': per_cat_mlm,
}).round(3)
comp.to_csv(OUT_DIR / 'crows_pairs_independence_per_category.csv')
print('\nPer-category comparison:')
print(comp.to_string())

Per-pair correlation:
  Pearson  r = -0.011
  Spearman r = -0.094
  n          = 1508

Per-category comparison:
                     DirAcc_ours  StereoScore_BERT_base
bias_type                                              
age                        0.554                  0.609
disability                 0.700                  0.783
gender                     0.543                  0.573
nationality                0.654                  0.465
physical-appearance        0.578                  0.698
race-color                 0.577                  0.597
religion                   0.707                  0.733
sexual-orientation         0.640                  0.774
socioeconomic              0.651                  0.587


## 12. Files written

In [ ]:
print('Files saved to:', OUT_DIR)
for f in sorted(OUT_DIR.glob('*.csv')):
    print(' -', f.name, f'({f.stat().st_size//1024} KB)')